In [1]:
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder


df = pd.read_csv('C:/Users/Lenovo/OneDrive/Desktop/BinX training/Project/data/heart_disease_clean.csv')


#  Prepare the data
X = df.drop(columns=['Result'])   
y = df['Result']

le = LabelEncoder()
y = le.fit_transform(y)   # encode positive -> 1, negative -> 0 

# Split into train / validation / test
X_train_full, X_test, y_train_full, y_test = train_test_split( X, y, test_size=0.15, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42, stratify=y_train_full)

# Scale AFTER splitting (fit only on train, to avoid data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

n_features = X_train.shape[1]

# --- Build the model ---
model = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),   # binary output
])

model.summary()

c:\Users\Lenovo\anaconda3\envs\tf_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,689 (10.50 KB)

 Trainable params: 2,689 (10.50 KB)

 Non-trainable params: 0 (0.00 B)

In [2]:
from tensorflow.keras.layers import BatchNormalization, Dropout
# Experiment: Dropout = 0.5 instead of 0.3
model_exp1 = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.5),
    Dense(1, activation="sigmoid"),
])

model_exp1.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history_exp1 = model_exp1.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=32)

Epoch 1/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4968 - loss: 0.9744 - val_accuracy: 0.5714 - val_loss: 0.6873
Epoch 2/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5453 - loss: 0.8542 - val_accuracy: 0.6369 - val_loss: 0.6527
Epoch 3/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5747 - loss: 0.7662 - val_accuracy: 0.6369 - val_loss: 0.6379
Epoch 4/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6189 - loss: 0.7408 - val_accuracy: 0.6607 - val_loss: 0.6286
Epoch 5/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6168 - loss: 0.7251 - val_accuracy: 0.6905 - val_loss: 0.6189
Epoch 6/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6358 - loss: 0.6709 - val_accuracy: 0.7024 - val_loss: 0.6054
Epoch 7/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6442 - loss: 0.6684 - val_accuracy: 0.7024 - val_loss: 0.6010
Epoch 8/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6547 - loss: 0.6465 - val_accuracy: 0.7202 - val_loss:

In [4]:
from tensorflow.keras.optimizers import Adam
# Experiment: lower learning rate
model_exp2 = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

model_exp2.compile(optimizer=Adam(learning_rate=0.0005), loss="binary_crossentropy", metrics=["accuracy"])
history_exp2 = model_exp2.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=32)

Epoch 1/50


c:\Users\Lenovo\anaconda3\envs\tf_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5063 - loss: 0.9362 - val_accuracy: 0.5655 - val_loss: 0.6765
Epoch 2/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5747 - loss: 0.7530 - val_accuracy: 0.6250 - val_loss: 0.6636
Epoch 3/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6053 - loss: 0.7117 - val_accuracy: 0.6369 - val_loss: 0.6473
Epoch 4/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6168 - loss: 0.6916 - val_accuracy: 0.6429 - val_loss: 0.6332
Epoch 5/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6463 - loss: 0.6543 - val_accuracy: 0.6488 - val_loss: 0.6200
Epoch 6/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6526 - loss: 0.6761 - val_accuracy: 0.6726 - val_loss: 0.6111
Epoch 7/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6579 - loss: 0.6694 - val_accuracy: 0.6786 - val_loss: 0.5997
Epoch 8/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6632 - loss: 0.6415 - val_accuracy: 0.7083 - val_loss: 0.5870
Epo

In [ ]:
# Experiment: higher learning rate
model_exp2 = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

model_exp2.compile(optimizer=Adam(learning_rate=0.005), loss="binary_crossentropy", metrics=["accuracy"])
history_exp2 = model_exp2.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=32)

Epoch 1/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6474 - loss: 0.6595 - val_accuracy: 0.6369 - val_loss: 0.5929
Epoch 2/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6895 - loss: 0.5733 - val_accuracy: 0.6607 - val_loss: 0.5703
Epoch 3/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7084 - loss: 0.5428 - val_accuracy: 0.6667 - val_loss: 0.5612
Epoch 4/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7547 - loss: 0.5051 - val_accuracy: 0.6607 - val_loss: 0.5512
Epoch 5/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7379 - loss: 0.4855 - val_accuracy: 0.6726 - val_loss: 0.5289
Epoch 6/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7589 - loss: 0.4553 - val_accuracy: 0.6548 - val_loss: 0.5308
Epoch 7/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7589 - loss: 0.4584 - val_accuracy: 0.7560 - val_loss: 0.4786
Epoch 8/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8042 - loss: 0.3967 - val_accuracy: 0.6310 - val_loss

In [6]:
# Experiment: smaller network
model_exp3 = Sequential([
    Dense(32, activation="relu", input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(16, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

model_exp3.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history_exp3 = model_exp3.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=32)

Epoch 1/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4989 - loss: 0.9012 - val_accuracy: 0.5119 - val_loss: 0.7174
Epoch 2/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5274 - loss: 0.7984 - val_accuracy: 0.5714 - val_loss: 0.6964
Epoch 3/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5726 - loss: 0.7330 - val_accuracy: 0.5952 - val_loss: 0.6801
Epoch 4/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5895 - loss: 0.6992 - val_accuracy: 0.6071 - val_loss: 0.6627
Epoch 5/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6021 - loss: 0.6836 - val_accuracy: 0.6607 - val_loss: 0.6456
Epoch 6/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6400 - loss: 0.6496 - val_accuracy: 0.6845 - val_loss: 0.6318
Epoch 7/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6232 - loss: 0.6366 - val_accuracy: 0.6786 - val_loss: 0.6221
Epoch 8/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6432 - loss: 0.6245 - val_accuracy: 0.6964 - val_loss:

In [ ]:
# Early Stopping
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

model_final = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

model_final.compile(optimizer=Adam(learning_rate=0.005), loss="binary_crossentropy", metrics=["accuracy"])

history_final = model_final.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    callbacks=[es]
)

print(f"Training stopped at epoch: {len(history_final.history['loss'])}")

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6116 - loss: 0.7071 - val_accuracy: 0.6190 - val_loss: 0.6056
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6958 - loss: 0.5839 - val_accuracy: 0.6310 - val_loss: 0.5809
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6979 - loss: 0.5527 - val_accuracy: 0.6548 - val_loss: 0.5642
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7242 - loss: 0.5236 - val_accuracy: 0.7202 - val_loss: 0.5335
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7389 - loss: 0.5019 - val_accuracy: 0.7619 - val_loss: 0.5188
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7432 - loss: 0.4767 - val_accuracy: 0.7798 - val_loss: 0.4770
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7611 - loss: 0.4653 - val_accuracy: 0.7440 - val_loss: 0.4813
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7747 - loss: 0.4440 - val_accuracy: 0.7560 - v

In [8]:
model_final.evaluate(X_test, y_test)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8737 - loss: 0.2909 


[0.2909039556980133, 0.8737373948097229]

## Sprint 1 Close-Out — Neural Network Tuning

**Reference point:**

| Model | Accuracy |
|---|---|
| Logistic Regression (baseline) | 0.792 |

**Hyperparameter experiments** (validation set, one variable changed at a time vs. the Dropout+BatchNorm model):

| Experiment | Change | Val Accuracy | Val Loss | vs. Baseline (0.792) |
|---|---|---|---|---|
| exp1 | Dropout = 0.5 | 0.887 | 0.318 | +0.095 |
| exp2 | Learning rate = 0.0005 (lower) | 0.833 | 0.400 | +0.041 |
| exp3 | Learning rate = 0.005 (higher) | 0.9405 | 0.187 | +0.149 |
| exp4 | Smaller network (32→16) | 0.9167 | 0.259 | +0.125 |

**Final model:** Dropout + BatchNorm, Learning rate = 0.005, with `EarlyStopping` (`patience=5`, `restore_best_weights=True`). Training stopped automatically at epoch 28/100, saving 72 unnecessary epochs.

**Test set results — full comparison:**

| Model | Test Accuracy | Test Loss | vs. Baseline (0.792) |
|---|---|---|---|
| Logistic Regression (baseline) | 0.792 | — | — |
| NN v1 (no regularization) | 0.793 | 0.381 | +0.001 |
| NN v2 (Dropout + BatchNorm) | 0.889 | 0.246 | +0.097 |
| **NN Final (tuned + EarlyStopping)** | **0.874** | **0.291** | **+0.082** |

**Conclusion:** Every regularized neural network variant beat the Logistic Regression baseline by 8–10 accuracy points. The tuning experiments (val set) showed the higher learning rate (0.005) alone reaching the best validation score (0.9405), but on the untouched test set, the final tuned model with EarlyStopping (0.874) landed slightly *below* NN v2 (0.889) — a small, expected variance rather than a real regression, since the two models differ only in learning rate and stopping point. The main, consistent driver of improvement across the board was regularization (Dropout + BatchNorm) itself, not fine-grained hyperparameter tuning. EarlyStopping proved useful in practice, cutting training time (28 vs. 100 epochs) without meaningfully hurting performance.